In [ ]:
"""
step1_bias_detector.py
Dataset Bias Detector and Corrector - Responsible AI.
Python 3.10+ | Random Forest | Target Accuracy > 95%
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import (train_test_split, cross_val_score,
StratifiedKFold)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
confusion_matrix, roc_auc_score,
precision_score, recall_score, f1_score)
# STEP 1 - DATA LOADING
df = pd.read_csv('loan_bias_dataset.csv')
print(f"""Records:{len(df):,} Features:{len(df.columns)-2}
Missing:{df.isnull().sum().sum()}""")
# STEP 2 - EXPLORATORY BIAS ANALYSIS
p_male = df[df['gender']==1]['loan_approved'].mean()
p_female = df[df['gender']==0]['loan_approved'].mean()
print(f"Approval - Male : {p_male:.4f} ({p_male*100:.2f}%)")
print(f"Approval - Female: {p_female:.4f} ({p_female*100:.2f}%)")
# STEP 3 - BIAS METRICS (BEFORE CORRECTION)
di_before = p_female / p_male
dpd_before = p_female - p_male
print(f"DIW (before): {di_before:.4f}")
print(f"DPD (before): {dpd_before:.4f}")
# STEP 4 - BIAS DETECTION DECISION GATE
biased = (di_before < 0.8) or (abs(dpd_before) > 0.1)
print(f"Dataset biased: {biased} - {"Proceeding to correction." if biased else "No action needed."}")
# STEP 5 - REWEIGHING ALGORITHM
def compute_reweighing_weights(df, sensitive_col, label_col,
                               priv_val=1, unpriv_val=0, fav_val=1):
    """
    W(g,y) = P(group=g) * P(label=y) / P(group=g, label=y)
    Normalized so mean weight = 1.
    Higher weights -> underrepresented (group, label) combinations.
    """
    n = len(df)
    weights = np.ones(n)
    for g in [priv_val, unpriv_val]:
        for y in [fav_val, 1 - fav_val]:
            mask = (df[sensitive_col]==g) & (df[label_col]==y)
            p_g = (df[sensitive_col]==g).mean()
            p_y = (df[label_col]==y).mean()
            p_gy = mask.mean()

            w = (p_g * p_y / p_gy) if p_gy > 0 else 1.0
            weights[mask.values] = w
    return weights * (n / weights.sum())
sample_weights = compute_reweighing_weights(df,'gender','loan_approved')
print(f"""Weights - Min:{sample_weights.min():.4f} Max:{sample_weights.max():.4f}
Mean:{sample_weights.mean():.4f}""")
# STEP 6 - POST-CORRECTION METRICS
def wtd_rate(df, w, g_col, g_val, y_col, y_val=1):
    mask = df[g_col] == g_val
    return (w[mask]*(df.loc[mask,y_col]==y_val)).sum() / w[mask].sum()
wt_male = wtd_rate(df, sample_weights,'gender',1,'loan_approved')
wt_female = wtd_rate(df, sample_weights,'gender',0,'loan_approved')
di_after = wt_female / wt_male
dpd_after = wt_female - wt_male
print(f"DIW (after): {di_after:.4f}") # -> 1.0000
print(f"DPD (after): {dpd_after:.4f}") # -> 0.0000
# STEP 7 - MODEL TRAINING
FEAT = ['age','education_level','annual_income','credit_score',
'employment_years','debt_ratio','loan_amount','dependents',
'prior_default','property_owner','savings_ratio']
X = df[FEAT].values; y_biased = df['loan_approved'].values
y_fair = df['fair_label'].values; w = sample_weights
idx_tr,idx_te = train_test_split(np.arange(len(df)),
test_size=0.20, random_state=42, stratify=y_fair)
X_tr,X_te = X[idx_tr], X[idx_te]
y_tr_fair,y_te_fair = y_fair[idx_tr], y_fair[idx_te]
y_tr_bias = y_biased[idx_tr]; w_tr = w[idx_tr]
RF = dict(n_estimators=500,max_depth=None,min_samples_leaf=1,
max_features="sqrt",random_state=42,n_jobs=-1)
# Model A - biased labels, no reweighing (baseline)
rf_orig = RandomForestClassifier(**RF)
rf_orig.fit(X_tr, y_tr_bias)
acc_orig = accuracy_score(y_te_fair, rf_orig.predict(X_te))
auc_orig = roc_auc_score(y_te_fair, rf_orig.predict_proba(X_te)[:,1])
# Model B - fair labels + reweighing (de-biased)
rf_rw = RandomForestClassifier(**RF)
rf_rw.fit(X_tr, y_tr_fair, sample_weight=w_tr)
pred_rw = rf_rw.predict(X_te)
acc_rw = accuracy_score(y_te_fair, pred_rw)
auc_rw = roc_auc_score(y_te_fair, rf_rw.predict_proba(X_te)[:,1])
prec_rw = precision_score(y_te_fair, pred_rw)
rec_rw = recall_score(y_te_fair, pred_rw)
f1_rw = f1_score(y_te_fair, pred_rw)
print(f"Model A Accuracy : {acc_orig*100:.2f}%")
print(f"Model B Accuracy : {acc_rw*100:.2f}% ROC-AUC: {auc_rw:.4f}")
# STEP 8 - 5-FOLD CROSS-VALIDATION
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvs = cross_val_score(rf_rw, X, y_fair, cv=cv, scoring="accuracy")
cva = cross_val_score(rf_rw, X, y_fair, cv=cv, scoring="roc_auc")
print(f"CV Accuracy: {cvs.mean()*100:.2f}% +/- {cvs.std()*100:.2f}%")
print(f"CV AUC : {cva.mean():.4f} +/- {cva.std():.4f}")
# STEP 9 - CLASSIFICATION REPORT & CONFUSION MATRIX
print(classification_report(y_te_fair, pred_rw,
target_names=["Denied (0)","Approved (1)"]))
cm = confusion_matrix(y_te_fair, pred_rw)
print(f"TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
# STEP 10 - FEATURE IMPORTANCE
fi = pd.DataFrame({"Feature":FEAT,"Importance":rf_rw.feature_importances_})
fi = fi.sort_values("Importance",ascending=False)
print(fi.to_string(index=False))

# STEP 11 - VISUALISATION DASHBOARD
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Dataset Bias Detector & Corrector - Results Dashboard",
fontsize=14, fontweight="bold")
# Panel 1 - Fairness Metrics
ax1=axes[0,0]; xp=np.arange(2)
ax1.bar(xp-0.2,[di_before,abs(dpd_before)],0.35,label="Before",color="#E74C3C")
ax1.bar(xp+0.2,[di_after, abs(dpd_after)], 0.35,label="After", color="#27AE60")
ax1.axhline(0.8,color="black",linestyle="--",label="Threshold(0.8)")
ax1.set_xticks(xp); ax1.set_xticklabels(["DIW","|DPD|"])
ax1.set_title("Fig 6.1 - Fairness Metrics Before vs. After"); ax1.legend()
# Panel 2 - Model Performance
ax2=axes[0,1]; xp2=np.arange(2)
ax2.bar(xp2-0.2,[acc_orig*100,acc_rw*100],0.35,label="Accuracy (%)",
color="#2980B9")
ax2.bar(xp2+0.2,[auc_orig*100,auc_rw*100],0.35,label="ROC-AUC (%)",color="#8E44AD")
ax2.axhline(95,color="orange",linestyle="--",label="95% Target")
ax2.set_xticks(xp2); ax2.set_xticklabels(["Original","De-biased"])
ax2.set_title("Fig 6.2 - Model Performance"); ax2.legend()
# Panel 3 - Gender Approval Rate
ax3=axes[1,0]; xp3=np.arange(2)
ax3.bar(xp3-0.2,[p_male*100,p_female*100], 0.35,label="Before",color="#E74C3C")
ax3.bar(xp3+0.2,[wt_male*100,wt_female*100],0.35,label="After", color="#27AE60")
ax3.set_xticks(xp3); ax3.set_xticklabels(["Male","Female"])
ax3.set_title("Fig 6.3 - Approval Rate by Gender"); ax3.legend()
# Panel 4 - Feature Importance
ax4=axes[1,1]
ax4.barh(fi["Feature"][::-1], fi["Importance"][::-1], color="#2980B9")
ax4.set_title("Fig 6.4 - Feature Importance (De-biased RF)")
ax4.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig("bias_correction_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
# STEP 12 - EXPORT DE-BIASED DATASET
df_out = df.drop(columns=["fair_label"]).copy()
df_out["sample_weight"] = np.round(sample_weights, 6)
df_out.to_csv("loan_debiased_dataset.csv", index=False)

Records:5,000 Features:12
Missing:0
Approval - Male : 0.6659 (66.59%)
Approval - Female: 0.0373 (3.73%)
DIW (before): 0.0560
DPD (before): -0.6286
Dataset biased: True - Proceeding to correction.
Weights - Min:0.5748 Max:10.2627
Mean:1.0000
DIW (after): 1.0000
DPD (after): 0.0000
Model A Accuracy : 71.50%
Model B Accuracy : 95.80% ROC-AUC: 0.9935
CV Accuracy: 96.06% +/- 0.81%
CV AUC : 0.9943 +/- 0.0016
              precision    recall  f1-score   support

  Denied (0)       0.95      0.92      0.94       332
Approved (1)       0.96      0.97      0.97       668

    accuracy                           0.96      1000
   macro avg       0.96      0.95      0.95      1000
weighted avg       0.96      0.96      0.96      1000

TN=307 FP=25 FN=17 TP=651
         Feature  Importance
    credit_score    0.291401
   annual_income    0.238401
 education_level    0.156776
employment_years    0.087484
             age    0.081367
   prior_default    0.037585
   savings_ratio    0.030417
      deb